# Hyperparameter Tuning with Optuna

This notebook performs hyperparameter optimization for selected models
(CatBoost, XGBoost, and Extra Trees) using Optuna.

The objective metric is **Average Precision (PR-AUC)**, which is more
appropriate for highly imbalanced classification problems such as fraud detection.

Sampling techniques and threshold tuning are intentionally excluded
at this stage to isolate the intrinsic performance of each model.


In [1]:
from datapipeline.data.load_data import load_raw_data
from datapipeline.config.mlflow_config import setup_mlflow
import yaml
import pandas as pd
import mlflow
from pathlib import Path
import optuna
from sklearn.metrics import (roc_auc_score, 
                            balanced_accuracy_score,
                            precision_score,
                            recall_score,
                            f1_score,
                            confusion_matrix,
                            precision_recall_curve,
                            average_precision_score,
                            make_scorer)
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from lightgbm import LGBMClassifier
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight


In [2]:
#primary metric that will be used for model comparison
PRIMARY_METRIC = "Average Precision Score"


In [3]:
#loading config file
config_path = '../config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

In [37]:
random_state = config["model_training"]["random_state"]


42

In [59]:
artifacts_dir = Path(config['model_selection']['artifacts_path'])

# MLflow

In [4]:
experiment_name = config['pipeline']['experiment_name']


In [5]:
setup_mlflow(experiment_name)

In [6]:
artifacts_dir = Path(config['model_tuning']['artifacts_path'])

# Load Dataset

In [7]:
dataset_path = Path(config['feature_engineering']['train_path_feature_engineered'])
target_column = config['data']['target_column']

In [8]:
df_train = pd.read_parquet(dataset_path)

In [9]:
y_train = df_train[target_column]

In [10]:
x_train = df_train.drop(columns = ['Time', target_column])

# Models

Models selected in the previous sted:

- Catboost
- Extra Trees
- Xgboost

# Cross Validation setup

In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=config["model_training"]["random_state"]
)

scoring = {
    'average_precision': 'average_precision'
          }

# Optuna objective — XGBoost

In [12]:
n_major = sum(y_train==0)
n_minor = sum(y_train==1)
scale_pos_weight = n_major / n_minor

In [13]:
def calculating_cross_validation(model):
    cv_results = cross_validate(model, 
                                x_train, 
                                y_train, 
                                cv=cv, 
                                scoring=scoring, 
                                verbose=False)
    return cv_results
    

In [ ]:
xgb.XGBClassifier(

In [14]:
def objective_xgboost(trial):

     params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 20.0),
        "random_state": random_state,
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": "aucpr",
        "tree_method": "hist"
     }

     model = xgb.XGBClassifier(**params)

     '''
     cv_results = cross_validate(model, 
                                x_train, 
                                y_train, 
                                cv=cv, 
                                scoring=scoring, 
                                verbose=False)
     '''
     cv_results = calculating_cross_validation(model)
     metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]

     return metrics_results

In [ ]:
with mlflow.start_run(run_name="xgboost_hyperparameter_tuning"):
    study = optuna.create_study(direction='maximize')
    study.optimize(objective_xgboost, n_trials=50)

    model = xgb.XGBClassifier(scale_pos_weight=scale_pos_weight)
    cv_results = calculating_cross_validation(model)
    xgb_best_params = study.best_params
    metric_default_hyperparameters = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]

    mlflow.log_metric('average_precision_default_params', metric_default_hyperparameters[0])
    mlflow.log_metric('best_cv_average_precision', study.best_value)
    mlflow.log_params(study.best_params)

print(f'Best average precision score: {study.best_value}')
print(f'Best parameters: {study.best_params}')

# Optuna objective — CatBoost

In [38]:
def objective_catboost(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 300, 800),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_state": random_state,
        "verbose": False
    }

    model = CatBoostClassifier(**params)

    cv_results = calculating_cross_validation(model)
    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]

    return metrics_results

In [ ]:
with mlflow.start_run(run_name="Catboost_hyperparameter_tuning"):
    study = optuna.create_study(direction='maximize')
    study.optimize(objective_catboost, n_trials=50)

    model = CatBoostClassifier()
    cv_results = calculating_cross_validation(model)
    metric_default_hyperparameters = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    catboost_best_params = study.best_params

    mlflow.log_metric('average_precision_default_params', metric_default_hyperparameters[0])
    mlflow.log_metric('best_cv_average_precision', study.best_value)
    mlflow.log_params(study.best_params)
    
print(f'Best average precision score: {study.best_params}')
print(f'Best parameters: {study.best_params}')

# Optuna objective — Extra Trees

In [20]:
def objective_extra_trees(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
        "class_weight": "balanced",
        "random_state": config["model_training"]["random_state"],
        "n_jobs": -1
    }

    model = ExtraTreesClassifier(**params)

    cv_results = calculating_cross_validation(model)

    metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]

    return metrics_results

In [ ]:
with mlflow.start_run(run_name="ExtraTrees_hyperparameter_tuning"):
    study = optuna.create_study(direction='maximize')
    study.optimize(objective_extra_trees, n_trials=50)

    model = ExtraTreesClassifier()
    cv_results = calculating_cross_validation(model)
    metric_default_hyperparameters = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
    extra_trees_best_params = study.best_params
    
    mlflow.log_metric('average_precision_default_params', metric_default_hyperparameters[0])
    mlflow.log_metric('best_cv_average_precision', study.best_value)
    mlflow.log_params(study.best_params)
    
print(f'Best average precision score: {study.best_params}')
print(f'Best parameters: {study.best_params}')

# Comparison using best hyperparameters

In [49]:
catboost_best_params['random_state'] = random_state
xgb_best_params['random_state'] = random_state
extra_trees_best_params['random_state'] = random_state
xgb_best_params["scale_pos_weight"] = scale_pos_weight

In [50]:
catboost = CatBoostClassifier(**catboost_best_params)
xgboost = xgb.XGBClassifier(**xgb_best_params)
extra_tree = ExtraTreesClassifier(**extra_trees_best_params)


In [51]:
candidate_models = {
                   'CatBoost': catboost,
                   'XgBoost': xgboost,
                   'Extra Trees': extra_tree
                    }

In [52]:
#metrics that will be used for model comparison
metrics = ['Balanced Accuracy Score',
           'Precision Score',
           'Recall Score',
           'F1 Score',
           'Average Precision Score',
           'Roc AUC']


In [53]:
scoring = {
    'balanced_accuracy': 'balanced_accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'average_precision': 'average_precision',
    'roc_auc': 'roc_auc'
}

In [60]:
comparison_best_hyperparameters = pd.DataFrame(columns = metrics,
                       index= candidate_models,
                       data = 0.0)

In [61]:
comparison_best_hyperparameters

,Balanced Accuracy Score,Precision Score,Recall Score,F1 Score,Average Precision Score,Roc AUC
CatBoost,0.0,0.0,0.0,0.0,0.0,0.0
XgBoost,0.0,0.0,0.0,0.0,0.0,0.0
Extra Trees,0.0,0.0,0.0,0.0,0.0,0.0


In [67]:
with mlflow.start_run(run_name="Model_comparison_using_tuned_hyperparameters"):
    for model_name, model in candidate_models.items():
        print(f'Training model {model_name}')
    
        cv_results = cross_validate(model, 
                                    x_train, 
                                    y_train, 
                                    cv=cv, 
                                    scoring=scoring, 
                                    verbose = False)
       
        metrics_results = [cv_results[f'test_{metric}'].mean() for metric in scoring.keys()]
        print(metrics_results)
        comparison_best_hyperparameters.loc[model_name, :] = metrics_results
    comparison_best_hyperparameters.to_parquet(
        artifacts_dir / "model_selection_tuned_hyperparameters.parquet")
    mlflow.log_artifact(
        artifacts_dir / "model_selection_tuned_hyperparameters.parquet",
        artifact_path="model_selection")

Training model CatBoost
0:	learn: 0.6174762	total: 26ms	remaining: 20.5s
1:	learn: 0.5568022	total: 34.8ms	remaining: 13.7s
2:	learn: 0.4931252	total: 60.5ms	remaining: 15.9s
3:	learn: 0.4415945	total: 99.2ms	remaining: 19.5s
4:	learn: 0.3917107	total: 209ms	remaining: 32.8s
5:	learn: 0.3481057	total: 249ms	remaining: 32.5s
6:	learn: 0.3090850	total: 309ms	remaining: 34.5s
7:	learn: 0.2725843	total: 348ms	remaining: 34s
8:	learn: 0.2439648	total: 380ms	remaining: 33s
9:	learn: 0.2163485	total: 409ms	remaining: 31.9s
10:	learn: 0.1916813	total: 434ms	remaining: 30.7s
11:	learn: 0.1702970	total: 460ms	remaining: 29.8s
12:	learn: 0.1510597	total: 486ms	remaining: 29s
13:	learn: 0.1352970	total: 515ms	remaining: 28.5s
14:	learn: 0.1203625	total: 549ms	remaining: 28.3s
15:	learn: 0.1080304	total: 578ms	remaining: 27.9s
16:	learn: 0.0973937	total: 605ms	remaining: 27.5s
17:	learn: 0.0866566	total: 633ms	remaining: 27.1s
18:	learn: 0.0773792	total: 659ms	remaining: 26.7s
19:	learn: 0.0692370	

In [68]:
comparison_best_hyperparameters

,Balanced Accuracy Score,Precision Score,Recall Score,F1 Score,Average Precision Score,Roc AUC
CatBoost,0.895811,0.960413,0.791678,0.867607,0.860184,0.984145
XgBoost,0.918403,0.906378,0.836952,0.869870,0.867680,0.978355
Extra Trees,0.891253,0.942505,0.782587,0.854783,0.857175,0.985061
